# Kamath × Menon × Siletti × ASAP scVI+scANVI Integration Explorer

Interactive exploration of outputs from `integrate_kamath_menon_siletti_asap.py`.

- **Section 1** — UMAP explorer: colour dots by any obs column. Use lasso-tool to select barcodes from DA neurons.
- **Section 2** — Label agreement heatmap: scANVI predictions vs native cell type labels
- **Section 3** - Read Depth QC — UMI counts & genes measured per cell type

In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
from scipy.cluster.hierarchy import linkage, leaves_list
from pathlib import Path

pio.renderers.default = 'plotly_mimetype'

### File paths

In [ ]:
repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / ".git").exists())

# ------------
# Folder name and string in h5ad with results from integrate_*.py
#
# Use downsampling_strategy = "asap96k_kamath97k_menon99k_siletti100k_hvg_u_da_markers" to load the H5AD
# from the run with all 4-sources and Kamath's DA markers included
downsampling_strategy = "asap96k_kamath97k_menon99k_siletti100k_hvg_u_da_markers" 
#
# Use downsampling_strategy = "asap96k_menon99k_siletti100k_list_asap96k_menon99k_siletti100k_hvg_minus_da_markers_gene_list_no_kamath_no_kamath"
# to load the H5AD from the run with 3-sources (excluding Kamath's data) and DA markers also excluded
# downsampling_strategy = "asap96k_menon99k_siletti100k_list_asap96k_menon99k_siletti100k_hvg_minus_da_markers_gene_list_no_kamath_no_kamath"
# ------------

h5ad_name = 'integrated_' + downsampling_strategy + '.h5ad'
OUTPUT_DIR = str(repo_root / "reference_building" / "da_neurons_identification_analysis" / "output" / downsampling_strategy)
REFERENCE_DATA_DIR = str(repo_root / "reference_building" / "da_neurons_identification_analysis" / "reference_data")  # Intermediate H5ADs used as inputs for integrate_kamath_menon_siletti_asap.py

## NOTE: Siletti datased labels provided by the Allen Institute (i.e including DA VGLUT2, not the ones at CELLxGENE)
REFERENCE_DATA_DIR_PATH = repo_root / "reference_building" / "da_neurons_identification_analysis" / "reference_data"
REFERENCE_DATA_DIR = str(REFERENCE_DATA_DIR_PATH)
ALLEN_CELL_METADATA_PATH = str(REFERENCE_DATA_DIR_PATH / "allen-brain-cell-atlas" / "metadata" / "WHB-10Xv3" / "20241115" / "cell_metadata.csv.gz")
ALLEN_MEMBERSHIP_PATH = str(REFERENCE_DATA_DIR_PATH / "allen-brain-cell-atlas" / "metadata" / "WHB-taxonomy" / "20240330" / "cluster_to_cluster_annotation_membership.csv.gz")

### Cell types and marker genes

In [ ]:
# ------------- KAMATH ANNOTATIONS -------------
# DA subtype vulnerability labels (Kamath et al., Nat Neurosci 2022, Figure 3A & 3B)
# Vulnerability statistical values were extracted from the main text.
_DA_PD_SUFFIXES = {
    'SOX6_AGTR1':        'SOX6_AGTR1_vulnerable',  # significantly vulnerable (OR = −0.82, FDR-adjusted P < 0.05)
    'SOX6_DDT':          'SOX6_DDT',
    'SOX6_PART1':        'SOX6_PART1_down_ns',     # on the vulnerable side, but not cited as significant (no full stats tables were found)
    'SOX6_GFRA2':        'SOX6_GFRA2_up_ns',       # on the resilent side, but not cited as significant (no full stats tables were found)
    'CALB1_CALCR':       'CALB1_CALCR',
    'CALB1_RBP4':        'CALB1_RBP4',
    'CALB1_PPP1R17':     'CALB1_PPP1R17',
    'CALB1_CRYM_CCDC68': 'CALB1_CRYM_CCDC68',
    'CALB1_GEM':         'CALB1_GEM_resistant',    # significantly resistant (OR = 1.46, FDR-adjusted P < 0.05)
    'CALB1_TRHR':        'CALB1_TRHR_resistant',   # significantly resistant (OR = 0.84, FDR-adjusted P < 0.05)
}

# Kamath's DA maker genes
_KAMATH_DA_MARKERS = (
    # Pan-DA
    'TH', 'SLC6A3', 'SLC18A2', 'DDC',
    # Main DA subtype markers
    'SOX6', 'CALB1',
    # Subtype-defining markers (see _DA_PD_SUFFIXES).
    'AGTR1', 'DDT', 'GFRA2', 'PART1', 'CALCR', 'CRYM', 'CCDC68', 'GEM', 'PPP1R17', 'RBP4', 'TRHR',
)

# ------------- EACH DATASET CELL TYPE LABELS -------------
# See "`obs` column reference for details"
_NATIVE_COLS_BY_SOURCE = {
    'kamath': ('cell_type_scanvi',       # Kamath cell type used by integrate_kamath_menon_siletti_asap.py to train scANVI for label propagation 
               'da_subtype'),            # DA-only subset of cell_type_scanvi
    'menon': ('menon_cell_type',         # Menon full cell type description
              'menon_author_cell_type'), # Abbreviated description. Maps 1:1 with menon_cell_type
    'siletti': (
        'siletti_cxg_cell_type', 'siletti_cxg_author_cell_type', # Siletti, originally deposited in CELLxGENE
        'siletti_allen_da_vglut2_label', 'siletti_allen_cluster', 'siletti_allen_cell_type', # Re-annotated by the Allen Institute (incl. "DA VGLUT2")
    ),
    'asap_cohort': ('pmdbs_sc_rnaseq.C_scANVI',   # scANVI-propagated labels from the ASAP pmdbs_sc_rnaseq pipeline (not from integrate_kamath_menon_siletti_asap.py)
                    'pmdbs_sc_rnaseq.cell_type'),  # MMC / Allen Brain Atlas labels from the ASAP pmdbs_sc_rnaseq pipeline (not from integrate_kamath_menon_siletti_asap.py)
}

### Load h5ad from integration

In [ ]:
# ── Load integrated AnnData (used by all sections below) ──────────────────
print('Loading integrated.h5ad ... see `reference_building/README.md`  for description of `obs`')
adata = ad.read_h5ad(os.path.join(OUTPUT_DIR, h5ad_name))
print(f'  {adata.n_obs:,} cells · {adata.n_vars:,} genes')
print(f'  Sources: {dict(adata.obs["source"].value_counts())}')
print(f'  obs columns: {adata.obs.columns.tolist()}')

In [ ]:
# <dataset>_DA obs columns: per-cell 'DA'/'non-DA' call using each source's
# OWN independent ground truth (not scANVI's cell_type_pred) -- for QC/
# benchmark use, not training. Each column applies only its own source's
# criterion; cells from other sources (and non-DA cells within the source)
# fall through to 'non-DA'.
#   menon_DA   -- menon_author_cell_type == 'Dopaminergic'
#   siletti_DA -- siletti_allen_da_vglut2 (boolean) == True. Uses the raw
#     boolean rather than the derived 'siletti_allen_da_vglut2_label' string
#     column, since that one isn't built until the next section below --
#     same underlying condition either way (True <-> 'DA VGLUT2').
#   kamath_DA  -- cell_type_scanvi is one of the 10 Kamath DA subtypes
#     (_DA_PD_SUFFIXES keys, e.g. 'SOX6_AGTR1', 'CALB1_GEM').
adata.obs['menon_DA'] = np.where(adata.obs['menon_author_cell_type'] == 'Dopaminergic', 'DA', 'non-DA')
adata.obs['siletti_DA'] = np.where(adata.obs['siletti_allen_da_vglut2'].astype(bool), 'DA', 'non-DA')
adata.obs['kamath_DA'] = np.where(adata.obs['cell_type_scanvi'].isin(_DA_PD_SUFFIXES), 'DA', 'non-DA')

for col in ('menon_DA', 'siletti_DA', 'kamath_DA'):
    print(adata.obs[col].value_counts().to_string())
    print()

In [ ]:
# Shared "Colour by" / crosstab-axis options per source.
_NATIVE_COLS_BY_SOURCE = {
    'kamath': ('cell_type_scanvi', 'da_subtype'),
    'menon': ('menon_cell_type', 'menon_author_cell_type'),
    'siletti': (
        'siletti_cxg_cell_type', 'siletti_cxg_author_cell_type', 'siletti_allen_da_vglut2_label',
        'siletti_allen_cluster', 'siletti_allen_cell_type',
    ),
    'asap_cohort': ('pmdbs_sc_rnaseq.C_scANVI', 'pmdbs_sc_rnaseq.cell_type'),
}


# subcluster -> cluster is a verified stable many-to-one mapping in Allen's taxonomy.
def _load_siletti_subcluster_to_cluster():
    membership = pd.read_csv(ALLEN_MEMBERSHIP_PATH, keep_default_na=False)
    sub = membership.loc[
        membership['cluster_annotation_term_set_name'] == 'subcluster',
        ['cluster_alias', 'cluster_annotation_term_name'],
    ].rename(columns={'cluster_annotation_term_name': 'subcluster_name'})
    clu = membership.loc[
        membership['cluster_annotation_term_set_name'] == 'cluster',
        ['cluster_alias', 'cluster_annotation_term_name'],
    ].rename(columns={'cluster_annotation_term_name': 'cluster_name'})
    merged = sub.merge(clu, on='cluster_alias').drop_duplicates('subcluster_name')
    return merged.set_index('subcluster_name')['cluster_name']


# Materialize Siletti's derived label columns once (NaN for non-Siletti cells).
if 'siletti_allen_da_vglut2' in adata.obs.columns:
    _siletti_mask = adata.obs['source'] == 'siletti'
    adata.obs['siletti_allen_da_vglut2_label'] = np.nan
    adata.obs.loc[_siletti_mask, 'siletti_allen_da_vglut2_label'] = (
        adata.obs.loc[_siletti_mask, 'siletti_allen_da_vglut2']
        .map({True: 'DA VGLUT2 (Allen)', False: 'non-DA'})
    )
    adata.obs['siletti_allen_cluster'] = np.nan
    adata.obs.loc[_siletti_mask, 'siletti_allen_cluster'] = (
        adata.obs.loc[_siletti_mask, 'siletti_allen_subcluster']
        .map(_load_siletti_subcluster_to_cluster())
    )
    # Coarse cell type -- abbreviated prefix of siletti_allen_cluster (e.g. 'Astro_60' -> 'Astro').
    adata.obs['siletti_allen_cell_type'] = (
        adata.obs['siletti_allen_cluster'].str.replace(r'_\d+$', '', regex=True)
    )
    print(
        f"  Derived siletti_allen_da_vglut2_label, siletti_allen_cluster, "
        f"siletti_allen_cell_type "
        f"({int(adata.obs['siletti_allen_cluster'].notna().sum()):,} Siletti cells)"
    )

In [ ]:
# Gene symbol -> Ensembl ID mapping, via Menon's 'feature_name' var column
# (covers every gene in the merged h5ad, since it's an inner join subset).
_menon_cache_paths = glob.glob(f'{REFERENCE_DATA_DIR}/menon/intermediate/*_sn_only*.h5ad')
if _menon_cache_paths:
    _menon_var = sc.read_h5ad(_menon_cache_paths[0], backed='r').var
    adata.var['gene_symbol'] = _menon_var['feature_name'].reindex(adata.var_names)
    print(f"  Gene symbols resolved for {adata.var['gene_symbol'].notna().sum():,}/{adata.n_vars:,} "
          f"genes (via {_menon_cache_paths[0]})")
else:
    adata.var['gene_symbol'] = np.nan
    print("  Warning: no Menon SN-only cache found -- gene_symbol lookup unavailable.")

In [ ]:
adata_wo_kamath = adata[adata.obs['source'] != "kamath"]
adata_kamath = adata[adata.obs['source'] == "kamath"]
adata_menon = adata[adata.obs['source'] == "menon"]
adata_siletti = adata[adata.obs['source'] == "siletti"]
adata_asap = adata[adata.obs['source'] == "asap"]

## 1. UMAP Explorer

Scatter plot of the scANVI-latent UMAP (`obsm['X_umap']`). Use the **Colour by** dropdown to switch between any obs column. Key analysis columns are listed first.

- Categorical / low-cardinality integer columns → discrete colour palette
- Continuous numeric columns → Viridis scale
- Cells with `NaN` in the selected column appear in light grey

Hover over a point to see its `source`, `cell_type_pred`, `cell_type_scanvi`, `da_subtype`, Menon native labels, and ASAP PMDBS sc-RNA-seq native labels.

In [ ]:
if 'X_umap' not in adata.obsm:
    raise KeyError(
        'X_umap not found in adata.obsm. '
        'Re-run integrate_kamath_menon_asap.py with --compute-umap.'
    )

umap_df = pd.DataFrame(
    adata.obsm['X_umap'],
    columns=['UMAP_1', 'UMAP_2'],
    index=adata.obs_names,
)
umap_df = pd.concat([umap_df, adata.obs], axis=1)

# Apply PD vulnerability suffixes for display (h5ad labels are unchanged)
for col in ('cell_type_pred', 'cell_type_scanvi', 'da_subtype'):
    if col in umap_df.columns:
        umap_df[col] = umap_df[col].replace(_DA_PD_SUFFIXES)

In [ ]:
def _plot_umaps_categ(
    adata,
    panel_colors=('source', 'cell_type_pred', 'da_subtype'),
    plot_w=400, plot_h=400,
    margin_l=60, margin_t=50, margin_b=50,
    legend_w=260, cbar_w=100,
    point_size=3, opacity=0.6,
):
    """
    3-panel interactive UMAP, colouring by obs columns. For gene expression, see _plot_umaps_by_gene().

    Lasso/box-select on any panel updates that panel's entry in the returned dict live --
    read it any time after selecting (no need to re-run the cell).

    Returns
    -------
    dict
        {panel_color: list} -- each list holds the obs_names (barcodes) currently
        lasso/box-selected in that panel. Empty until you make a selection.
    """
    umap_df = pd.DataFrame(
        adata.obsm['X_umap'], columns=['UMAP_1', 'UMAP_2'], index=adata.obs_names,
    )
    umap_df = pd.concat([umap_df, adata.obs], axis=1)
    umap_df['_barcode'] = umap_df.index  # customdata needs a column, not the index itself

    # PD vulnerability suffixes for display only (h5ad labels unchanged) — see _DA_PD_SUFFIXES.
    for col in ('cell_type_pred', 'cell_type_scanvi', 'da_subtype'):
        if col not in umap_df.columns:
            continue
        if isinstance(umap_df[col].dtype, pd.CategoricalDtype):
            rename = {k: v for k, v in _DA_PD_SUFFIXES.items() if k in umap_df[col].cat.categories}
            umap_df[col] = umap_df[col].cat.rename_categories(rename)
        else:
            umap_df[col] = umap_df[col].replace(_DA_PD_SUFFIXES)

    # Same priority columns as the Label Agreement widget (_NATIVE_COLS_BY_SOURCE).
    priority = ['source', 'cell_type_pred', 'cell_type_scanvi', 'da_subtype']
    for _cols in _NATIVE_COLS_BY_SOURCE.values():
        priority.extend(c for c in _cols if c not in priority)
    all_obs = adata.obs.columns.tolist()
    color_options = (
        [c for c in priority if c in all_obs]
        + sorted(c for c in all_obs if c not in priority)
    )
    hover_cols = [c for c in priority if c in umap_df.columns]

    def is_categorical(col):
        s = umap_df[col]
        return (
            pd.api.types.is_object_dtype(s)
            or pd.api.types.is_bool_dtype(s)
            or hasattr(s, 'cat')
            or (pd.api.types.is_integer_dtype(s) and s.nunique() < 30)
        )

    def build_figure(color_by, size, alpha, selected_barcodes, out):
        is_cat = is_categorical(color_by)
        r_margin = legend_w if is_cat else cbar_w
        total_w = margin_l + plot_w + r_margin

        hover = {c: True for c in hover_cols if c != color_by}
        common = dict(
            data_frame=umap_df, x='UMAP_1', y='UMAP_2',
            color=color_by, hover_data=hover, opacity=alpha,
            custom_data=['_barcode'], title=f'<b>{color_by}</b>',
        )
        fig = px.scatter(**common, color_continuous_scale=None if is_cat else 'Viridis')
        if is_cat:
            fig.update_layout(legend=dict(
                title=None, itemsizing='constant',
                x=1.01, xanchor='left', y=0.5, yanchor='middle',
            ))

        fig.update_traces(marker_size=size)
        fig.update_layout(
            autosize=False, width=total_w, height=plot_h + margin_t + margin_b,
            xaxis_title='UMAP 1', yaxis_title='UMAP 2',
            margin=dict(autoexpand=False, l=margin_l, r=r_margin, t=margin_t, b=margin_b),
            dragmode='lasso',  # click-drag lasso-selects by default; box-select still in the toolbar
        )

        fig = go.FigureWidget(fig)

        # A categorical `color` builds one trace per category, so point_inds are
        # trace-local -- customdata (the barcode) stays point-aligned regardless,
        # so read barcodes from it directly instead of re-deriving row positions.
        per_trace_selection = [[] for _ in fig.data]

        def make_on_select(trace_idx):
            def on_select(trace, points, selector):
                per_trace_selection[trace_idx] = [
                    trace.customdata[i][0] for i in points.point_inds
                ]
                selected_barcodes[:] = [b for sel in per_trace_selection for b in sel]
                with out:
                    print(f"  [{color_by}] {len(selected_barcodes):,} cells selected")
            return on_select

        for i, trace in enumerate(fig.data):
            trace.on_selection(make_on_select(i))

        return fig

    def make_panel(default_color):
        if default_color not in color_options:
            print(f"  Warning: '{default_color}' not in adata.obs — defaulting to '{color_options[0]}'")
            default_color = color_options[0]

        color_w = widgets.Dropdown(
            options=color_options, value=default_color,
            description='Colour by:', style={'description_width': 'initial'},
            layout=widgets.Layout(width='340px'),
        )
        size_w = widgets.IntSlider(
            value=point_size, min=1, max=10, step=1, description='Point size:',
            layout=widgets.Layout(width='320px'),
        )
        opacity_w = widgets.FloatSlider(
            value=opacity, min=0.1, max=1.0, step=0.05, description='Opacity:',
            layout=widgets.Layout(width='320px'),
        )
        out = widgets.Output()
        selected_barcodes = []

        def redraw(_=None):
            out.clear_output(wait=True)
            with out:
                display(build_figure(color_w.value, size_w.value, opacity_w.value, selected_barcodes, out))

        color_w.observe(redraw, names='value')
        size_w.observe(redraw, names='value')
        opacity_w.observe(redraw, names='value')

        vbox = widgets.VBox([color_w, size_w, opacity_w, out])

        def init():
            with out:
                display(build_figure(color_w.value, size_w.value, opacity_w.value, selected_barcodes, out))

        return vbox, init, default_color, selected_barcodes

    panels = [make_panel(c) for c in panel_colors]
    for _, init, _, _ in panels:
        init()
    display(widgets.HBox([box for box, _, _, _ in panels]))

    return {color: selected_barcodes for _, _, color, selected_barcodes in panels}

### Colour UMAP plots by cell types

In [ ]:
# Show UMAP and save lasso-selected barcode ID's to `selected_barcodes`
selected_barcodes = _plot_umaps_categ(adata=adata)

### Save list of barcodes selected with lasso tool

In [ ]:
for colourby in selected_barcodes.keys():
    if len(selected_barcodes[colourby]) > 0:
        out_path = Path(OUTPUT_DIR) / f'selected_barcodes_{colourby}.txt'
        with open(out_path, 'w') as f:
            for barcode in selected_barcodes[colourby]:
                f.write(barcode + '\n')

        print(f"Wrote {len(selected_barcodes[colourby]):,} barcodes -> {out_path}")

In [ ]:
_plot_umaps_categ(adata=adata_siletti)

In [ ]:
_plot_umaps_categ(adata=adata_menon)

In [ ]:
_plot_umaps_categ(adata=adata_kamath)

In [ ]:
_plot_umaps_categ(adata=adata_asap)

### 1b. Gene Expression UMAP

Same UMAP coordinates as above, but each panel has a free-text **Gene** box instead of a dropdown — type any gene symbol (or Ensembl ID) present in this h5ad's gene panel and it colours the plot by that gene's log-normalized expression (continuous Viridis scale). Not limited to a predefined list: resolution goes through `adata.var['gene_symbol']` (built from Menon's native panel, covers every gene in the merged h5ad) via `sc.get.obs_df(..., gene_symbols=...)`, scanpy's own mechanism for this.

The suggested/autocomplete options are the marker genes Kamath et al. (Nat Neurosci 2022) used to identify and subtype DA neurons — but typing anything else works too:

- **Pan-DA identity markers** (validate broad dopaminergic identity, not subtype-specific): `TH`, `SLC6A3` (DAT), `SLC18A2` (VMAT2), `DDC` (AADC)
- **Branch markers** (the two main DA subdivisions Kamath split subtypes on): `SOX6`, `CALB1`
- **Subtype-defining markers** (combined with SOX6/CALB1 to name each of the 10 Kamath subtypes, e.g. `SOX6_AGTR1`, `CALB1_GEM`): `AGTR1`, `DDT`, `GFRA2`, `PART1`, `CALCR`, `CRYM`, `CCDC68`, `GEM`, `PPP1R17`, `RBP4`, `TRHR`

A gene not present in this h5ad's gene panel (e.g. ASAP's own upstream gene panel is only ~3,000 genes, so a 4-source inner join can drop markers ASAP never measured) shows a message in the panel instead of failing silently.

First panel set is on Kamath cells only (known ground-truth identity — expect `TH`/`SLC6A3`/`SLC18A2`/`DDC` high throughout, and a clean `SOX6` vs `CALB1` split). Second is on all cells, to see where marker expression concentrates across the other three sources relative to `cell_type_pred`.

In [ ]:
def _plot_umaps_by_gene(
    adata,
    panel_genes=None,
    n_panels=4,
    suggested_genes=_KAMATH_DA_MARKERS,
    plot_w=320, plot_h=320,
    margin_l=55, margin_t=50, margin_b=50, cbar_w=90,
    point_size=3, opacity=0.6,
):
    """N-panel UMAP with a free-text gene box per panel; colours by log-normalized expression of any gene."""
    if panel_genes is None:
        panel_genes = ('',) * n_panels

    umap_coords = pd.DataFrame(
        adata.obsm['X_umap'], columns=['UMAP_1', 'UMAP_2'], index=adata.obs_names,
    )

    def resolve_expression(query):
        """Return (expression Series, display label) or (None, None) if not found."""
        query = query.strip()
        if not query:
            return None, None
        try:
            df = sc.get.obs_df(adata, keys=[query.upper()], gene_symbols='gene_symbol')
            return df[query.upper()], query.upper()
        except KeyError:
            pass
        if query in adata.var_names:
            symbol = adata.var.loc[query, 'gene_symbol']
            label = symbol if pd.notna(symbol) else query
            return sc.get.obs_df(adata, keys=[query])[query], label
        return None, None

    def build_figure(gene_query, size, alpha):
        total_w = margin_l + plot_w + cbar_w
        expr, label = resolve_expression(gene_query)
        if expr is None:
            fig = go.Figure()
            if gene_query.strip():
                text = f"'{gene_query}' not found in this h5ad's gene panel ({adata.n_vars:,} genes)"
                color = 'red'
            else:
                text = "Type a gene symbol or Ensembl ID above"
                color = 'gray'
            fig.add_annotation(
                text=text,
                xref='paper', yref='paper', x=0.5, y=0.5, showarrow=False,
                font=dict(size=13, color=color),
            )
            fig.update_layout(width=total_w, height=plot_h + margin_t + margin_b)
            return fig

        df = umap_coords.copy()
        df['expression'] = expr.values
        # sort_order equivalent (as in sc.pl.umap): draw highest-expressing
        # cells last (on top) so they don't get buried under low/zero-
        # expression cells -- plotly renders scatter points in row order,
        # later rows on top. NaNs sorted first so they never cover real values.
        df = df.sort_values('expression', ascending=True, na_position='first')

        fig = px.scatter(
            df, x='UMAP_1', y='UMAP_2', color='expression',
            color_continuous_scale='Viridis', opacity=alpha,
            title=f'<b>{label}</b>',
        )
        fig.update_traces(marker_size=size)
        fig.update_layout(
            autosize=False, width=total_w, height=plot_h + margin_t + margin_b,
            xaxis_title='UMAP 1', yaxis_title='UMAP 2',
            coloraxis_colorbar=dict(title='log-norm expr'),
            margin=dict(autoexpand=False, l=margin_l, r=cbar_w, t=margin_t, b=margin_b),
        )
        return fig

    def make_panel(default_gene):
        gene_w = widgets.Combobox(
            value=default_gene, options=list(suggested_genes), ensure_option=False,
            description='Gene:', style={'description_width': 'initial'},
            layout=widgets.Layout(width='280px'), continuous_update=False,
        )
        size_w = widgets.IntSlider(
            value=point_size, min=1, max=10, step=1, description='Point size:',
            layout=widgets.Layout(width='260px'),
        )
        opacity_w = widgets.FloatSlider(
            value=opacity, min=0.1, max=1.0, step=0.05, description='Opacity:',
            layout=widgets.Layout(width='260px'),
        )
        out = widgets.Output()

        def redraw(_=None):
            out.clear_output(wait=True)
            with out:
                display(build_figure(gene_w.value, size_w.value, opacity_w.value))

        gene_w.observe(redraw, names='value')
        size_w.observe(redraw, names='value')
        opacity_w.observe(redraw, names='value')

        vbox = widgets.VBox([gene_w, size_w, opacity_w, out])

        def init():
            with out:
                display(build_figure(gene_w.value, size_w.value, opacity_w.value))

        return vbox, init

    panels = [make_panel(g) for g in panel_genes]
    for _, init in panels:
        init()
    display(widgets.HBox([box for box, _ in panels]))  

### Colour UMAP plots by gene expression

In [ ]:
_plot_umaps_by_gene(adata=adata_kamath, panel_genes=('TH', 'CALB1', 'SOX6', 'AGTR1'))

## 2. Label Agreement

In [ ]:
# DA-related prefixes highlighted red in plot labels.
_DA_LABEL_PREFIXES = ('CALB1_', 'SOX6_', 'DA VGLUT2', 'Splat_395', 'Dopaminergic')


def _colorize_label(label, prefixes=_DA_LABEL_PREFIXES, color='red'):
    """Wrap `label` in a colored HTML span if it starts with any of `prefixes`."""
    if str(label).startswith(prefixes):
        return f'<span style="color:{color}">{label}</span>'
    return label


def _hierarchical_leaf_order(matrix_df):
    """Leaf order via average-linkage clustering on row-normalized profiles."""
    if matrix_df.shape[0] < 3:
        return matrix_df.index
    row_sums = matrix_df.sum(axis=1).replace(0, 1)
    profiles = matrix_df.div(row_sums, axis=0)
    link = linkage(profiles.values, method='average', metric='euclidean')
    return matrix_df.index[leaves_list(link)]


def _plot_crosstab_heatmap(
    obs_df, row_col, col_col, normalize, show_values=False,
    color_scale='Blues', remap_da=True, sort_by='counts (max to min)',
    marginal_frac=0.10,
):
    """Cross-tabulate two obs columns as an interactive heatmap with log10 marginal histograms."""
    remap = _DA_PD_SUFFIXES if remap_da else {}
    pair = pd.DataFrame({
        'rows': obs_df[row_col].map(lambda x: remap.get(x, x)),
        'cols': obs_df[col_col],
    }).dropna()

    if pair.empty:
        print(f'No non-null pairs for ({row_col}, {col_col}).')
        return

    raw = pd.crosstab(pair['rows'], pair['cols']).astype(float)

    if sort_by == 'alphanumeric':
        row_order = raw.index.sort_values()
        col_order = raw.columns.sort_values()
    elif sort_by == 'cluster':
        row_order = _hierarchical_leaf_order(raw)
        col_order = _hierarchical_leaf_order(raw.T)
    else:  # 'counts (max to min)'
        row_order = raw.sum(axis=1).sort_values(ascending=False).index
        col_order = raw.sum(axis=0).sort_values(ascending=False).index

    ct = raw.copy()
    if normalize == 'rows (%)':
        ct = ct.div(ct.sum(axis=1), axis=0) * 100
        colorbar_title = 'rows (%)'
        text_fmt = '.1f'
    elif normalize == 'columns (%)':
        ct = ct.div(ct.sum(axis=0), axis=1) * 100
        colorbar_title = 'columns (%)'
        text_fmt = '.1f'
    else:
        colorbar_title = 'cells (n)'
        text_fmt = '.0f'

    ct = ct.loc[row_order, col_order]
    row_totals = raw.sum(axis=1).loc[row_order]
    col_totals = raw.sum(axis=0).loc[col_order]

    marginal_px = round(140 * marginal_frac / 0.18)  # scale fixed padding with marginal_frac
    height = max(500, len(ct) * 18 + 220) + marginal_px
    width  = max(650, len(ct.columns) * 20 + 280) + marginal_px

    text = ct.map(lambda v: format(v, text_fmt)).values if show_values else None

    fig = make_subplots(
        rows=2, cols=2,
        shared_xaxes=True, shared_yaxes=True,
        row_heights=[1 - marginal_frac, marginal_frac],
        column_widths=[marginal_frac, 1 - marginal_frac],
        horizontal_spacing=0.01, vertical_spacing=0.02,
    )

    # Row/column marginals: log10(cell count) bars, between labels and heatmap.
    fig.add_trace(
        go.Bar(
            x=row_totals.values, y=row_totals.index, orientation='h',
            marker_color='#6B6470', customdata=row_totals.values,
            hovertemplate='%{y}: %{customdata:,.0f} cells<extra></extra>',
        ),
        row=1, col=1,
    )
    fig.add_trace(
        go.Bar(
            x=col_totals.index, y=col_totals.values,
            marker_color='#6B6470', customdata=col_totals.values,
            hovertemplate='%{x}: %{customdata:,.0f} cells<extra></extra>',
        ),
        row=2, col=2,
    )
    # Main heatmap (row=2, col=1 left empty). xgap/ygap draw a subtle grid.
    fig.add_trace(
        go.Heatmap(
            z=ct.values, x=ct.columns, y=ct.index,
            colorscale=color_scale,
            text=text, texttemplate='%{text}' if show_values else None,
            colorbar=dict(title=colorbar_title),
            hovertemplate=f'{row_col}=%{{y}}<br>{col_col}=%{{x}}<br>{colorbar_title}=%{{z}}<extra></extra>',
            xgap=1, ygap=1,
        ),
        row=1, col=2,
    )

    fig.update_xaxes(type='log', autorange='reversed', showticklabels=False, row=1, col=1)
    fig.update_yaxes(
        showticklabels=True, tickmode='array',
        tickvals=list(row_order), ticktext=[_colorize_label(l) for l in row_order],
        row=1, col=1,
    )
    fig.update_xaxes(showticklabels=False, row=1, col=2)
    fig.update_yaxes(showticklabels=False, row=1, col=2)
    fig.update_yaxes(type='log', autorange='reversed', showticklabels=False, row=2, col=2)
    fig.update_xaxes(
        showticklabels=True, tickangle=45, tickmode='array',
        tickvals=list(col_order), ticktext=[_colorize_label(l) for l in col_order],
        row=2, col=2,
    )

    # Axis titles placed a fixed pixel distance from the canvas edge (not a
    # fixed paper-fraction, which drifts off-canvas as the plot grows).
    margin_l, margin_b, margin_t, margin_r = 220, 150, 60, 60
    plot_h = height - margin_t - margin_b
    plot_w = width - margin_l - margin_r
    x_title_y = -(margin_b - 20) / plot_h
    y_title_x = -(margin_l - 20) / plot_w

    heatmap_x_center = (marginal_frac + 1) / 2
    heatmap_y_center = (marginal_frac + 1) / 2
    fig.add_annotation(
        text=f'<b>{col_col}</b>', xref='paper', yref='paper',
        x=heatmap_x_center, y=x_title_y, xanchor='center', yanchor='top',
        showarrow=False, font=dict(size=13),
    )
    fig.add_annotation(
        text=f'<b>{row_col}</b>', xref='paper', yref='paper',
        x=y_title_x, y=heatmap_y_center, xanchor='center', yanchor='middle',
        textangle=-90, showarrow=False, font=dict(size=13),
    )

    fig.update_layout(
        height=height, width=width,
        showlegend=False,
        margin=dict(l=margin_l, b=margin_b, t=margin_t, r=margin_r),
        title=f'{row_col}  ×  {col_col}  ({colorbar_title})  —  marginals: log10(cell count)',
    )
    display(fig)

In [ ]:
_COLOR_SCALE_BY_SOURCE = {
    'kamath': 'Purples',
    'menon': 'Blues',
    'siletti': 'Greens',
    'asap_cohort': 'Oranges',
}


def label_agreement_widget(adata, source, pred_cols=('cell_type_pred', 'cell_type_scanvi')):
    """Interactive predicted-vs-native label cross-tab heatmap, for one source."""
    obs = adata.obs[adata.obs['source'] == source]
    row_opts = [c for c in pred_cols if c in obs.columns]
    col_opts = [c for c in _NATIVE_COLS_BY_SOURCE.get(source, ()) if c in obs.columns]

    print(f'{source} cells: {len(obs):,}')
    if not row_opts:
        print(f'No prediction columns found for source={source!r}.')
        return
    if not col_opts:
        print(f'No native label columns found for source={source!r}.')
        return

    @widgets.interact(
        row_col=widgets.Dropdown(
            options=row_opts,
            description='Rows (y-axis):',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='420px'),
        ),
        col_col=widgets.Dropdown(
            options=col_opts,
            description='Columns (x-axis):',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='420px'),
        ),
        normalize=widgets.Dropdown(
            options=['rows (%)', 'columns (%)', 'raw counts'],
            description='Colour encodes:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='420px'),
        ),
        sort_by=widgets.Dropdown(
            options=['counts (max to min)', 'alphanumeric', 'cluster'],
            description='Sort by:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='420px'),
        ),
    )
    def _plot(row_col, col_col, normalize, sort_by):
        _plot_crosstab_heatmap(
            obs, row_col, col_col, normalize,
            color_scale=_COLOR_SCALE_BY_SOURCE.get(source),
            sort_by=sort_by,
        )

In [ ]:
label_agreement_widget(adata, source='kamath')

In [ ]:
label_agreement_widget(adata, source='siletti')

In [ ]:
label_agreement_widget(adata, source='menon')

## 3. Read Depth QC — UMI counts & genes measured per cell type

Per-cell UMI counts (total raw counts) and genes-measured (nonzero genes),
grouped by each source's own **native** cell type label, read straight from
that source's intermediate cache (native full transcriptome — not the
merged/HVG-restricted gene set the model trains on). This is what motivated
the Kamath overfitting investigation: native DA neurons carry ~4-5x the raw
UMI content of glial cell types in the same collection, before any of our
own processing touches the data.

For Siletti, an extra `DA VGLUT2 (Allen)` row is added — Siletti's own
CellXGene annotation never labels anything dopaminergic (see Section 2), so
this population only exists via the Allen Brain Cell Atlas cross-reference.

Picks the *largest* available intermediate cache per source (by cell count),
regardless of which downsample scale the currently-loaded `adata` used —
read-depth characteristics of a random subsample don't meaningfully differ
from the full pool they were drawn from.

In [ ]:
# Multiplies the base plot width (1500px).
BOXPLOT_WIDTH_RATIO = 0.5
ASAP_CUTOFF_N_GENES = 10000
ASAP_CUTOFF_UMI = 100000


_INTERMEDIATE_GLOBS = {
    'kamath': f'{REFERENCE_DATA_DIR}/kamath/intermediate/kamath_merged_da_all_non_da_*.h5ad',
    'menon': f'{REFERENCE_DATA_DIR}/menon/intermediate/*_sn_only*.h5ad',
    'siletti': f'{REFERENCE_DATA_DIR}/siletti/intermediate/midbrain_da_neurons*_cells*.h5ad',
    'asap_cohort': f'{REFERENCE_DATA_DIR}/asap_cohort/pmdbs_sc_rnaseq/cohort_analysis/*/*/intermediate/*_rand_*.h5ad',
}
_NATIVE_GROUP_COL_BY_SOURCE = {
    'kamath': 'cell_type_scanvi',
    'menon': 'menon_author_cell_type',
    'siletti': 'siletti_cxg_cell_type',
    'asap_cohort': 'pmdbs_sc_rnaseq.cell_type',
}


_SOURCE_KEY_BY_NAME = {'kamath': 'kamath', 'menon': 'menon', 'siletti': 'siletti', 'asap_cohort': 'asap'}


def _parse_downsampling_targets(downsampling):
    """Parse {'asap': 50000, 'kamath': 52000, ...} from 'asap50k_kamath52k_...'."""
    return {m.group(1): int(m.group(2)) * 1000 for m in re.finditer(r'([a-z]+)(\d+)k', downsampling)}


def _matching_intermediate_cache(source, downsampling):
    """Intermediate cache for `source` whose cell count best matches `downsampling`."""
    paths = glob.glob(_INTERMEDIATE_GLOBS[source])
    if not paths:
        return None
    if len(paths) == 1:
        return paths[0]
    target = _parse_downsampling_targets(downsampling).get(_SOURCE_KEY_BY_NAME[source])
    if target is None:
        return max(paths, key=lambda p: sc.read_h5ad(p, backed='r').n_obs)
    return min(paths, key=lambda p: abs(sc.read_h5ad(p, backed='r').n_obs - target))


def _boxplot_pair_by_celltype(per_cell_df, group_col, source, sort_by='median'):
    """UMI counts / genes measured box plots per group, DA labels in red. sort_by: 'median' or 'alphanumeric'."""
    if sort_by == 'alphanumeric':
        order = sorted(per_cell_df[group_col].unique())
    else:
        order = (
            per_cell_df.groupby(group_col, observed=True)['total_counts']
            .median().sort_values(ascending=False).index.tolist()
        )
    order = order[::-1]  # plotly renders categoryarray bottom-to-top

    fig = make_subplots(
        rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.03,
        subplot_titles=('UMI counts', 'genes measured'),
    )
    fig.add_trace(go.Box(
        x=per_cell_df['total_counts'], y=per_cell_df[group_col], orientation='h',
        boxpoints=False, line_width=1, marker_color='#3D1F61', fillcolor='#3D1F61',
    ), row=1, col=1)
    fig.add_trace(go.Box(
        x=per_cell_df['n_genes'], y=per_cell_df[group_col], orientation='h',
        boxpoints=False, line_width=1, marker_color='#D98E04', fillcolor='#D98E04',
    ), row=1, col=2)

    fig.update_yaxes(
        categoryorder='array', categoryarray=order,
        tickmode='array', tickvals=order, ticktext=[_colorize_label(l) for l in order],
        row=1, col=1,
    )
    fig.update_xaxes(type='log', row=1, col=1)
    fig.update_xaxes(type='log', row=1, col=2)
    height = max(400, len(order) * 18 + 150)
    width = round(1500 * BOXPLOT_WIDTH_RATIO)
    fig.update_layout(
        height=height, width=width, showlegend=False,
        title=f'{source}: read depth per cell type (native transcriptome)',
        margin=dict(l=260, t=80, b=40, r=40),
    )
    fig.add_vline(
        x=ASAP_CUTOFF_UMI, row=1, col=1,
        line_dash='dot', line_color='red', line_width=1.5,
        annotation_text=f'{ASAP_CUTOFF_UMI:,}', annotation_position='top',
        annotation_font_color='red',
    )
    fig.add_vline(
        x=ASAP_CUTOFF_N_GENES, row=1, col=2,
        line_dash='dot', line_color='red', line_width=1.5,
        annotation_text=f'{ASAP_CUTOFF_N_GENES:,}', annotation_position='top',
        annotation_font_color='red',
    )
    display(fig)


def qc_boxplot_widget(source, sort_by='median'):
    """UMI-count and genes-measured box plots per native cell type, for `source`."""
    path = _matching_intermediate_cache(source, downsampling_strategy)
    if path is None:
        print(f'No intermediate cache found for source={source!r}.')
        return
    group_col = _NATIVE_GROUP_COL_BY_SOURCE[source]
    print(f'Loading {path} ...')
    a = sc.read_h5ad(path)
    if group_col not in a.obs.columns:
        print(f'Column {group_col!r} not found in {path}.')
        return

    counts = a.layers['counts']
    df = pd.DataFrame({
        'total_counts': np.asarray(counts.sum(axis=1)).ravel(),
        'n_genes': np.asarray((counts > 0).sum(axis=1)).ravel(),
        group_col: a.obs[group_col].astype(str).values,
    })

    if source == 'siletti' and 'siletti_allen_da_vglut2' in a.obs.columns:
        da_mask = a.obs['siletti_allen_da_vglut2'].astype(bool).values
        da_rows = df[da_mask].copy()
        da_rows[group_col] = 'DA VGLUT2 (Allen)'
        df = pd.concat([df, da_rows], ignore_index=True)

    print(f'{source}: {a.n_obs:,} cells, {df[group_col].nunique()} groups')
    _boxplot_pair_by_celltype(df, group_col, source, sort_by=sort_by)

In [ ]:
@widgets.interact(
    source=widgets.Dropdown(
        options=['kamath', 'menon', 'siletti', 'asap_cohort'],
        description='Source:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px'),
    ),
    sort_by=widgets.Dropdown(
        options=['median', 'alphanumeric'],
        description='Sort by:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px'),
    ),
)
def _run_qc_boxplot(source, sort_by):
    qc_boxplot_widget(source, sort_by=sort_by)